In [1]:
import sys
import pandas as pd

print(sys.version)
print(pd.__version__)

3.14.6 (main, Aug  4 2026, 19:23:56) [Clang 22.1.3 ]
3.0.5


## 1. Load and look

In [2]:
loans_raw = pd.read_csv('data/loans.csv')
loans = loans_raw.copy()

print(loans.shape)

(623, 9)


In [3]:
loans.head()

,loan_id,member_id,title,genre,loan_date,return_date,late_fee,returned,notes
0,L-30001,M0150,A Second Spring,ROMANCE,2026-01-06,2026-01-30,"7,50 kr",Yes,NaN
1,L-30002,M0037,The Silent Harbour,Crime,2026-01-05,2026-01-25,"0,00 kr",YES,NaN
2,L-30003,M0143,The Glass Mage,fantasy,2026-01-05,2026-01-26,"0,00 kr",YES,NaN
3,L-30003,M0143,The Glass Mage,fantasy,2026-01-05,2026-01-26,"0,00 kr",YES,NaN
4,L-30004,M0121,Dead Reckoning Lane,CRIME,2026-01-05,2026-01-12,"0,00 kr",Y,NaN


In [4]:
loans.info()

<class 'pandas.DataFrame'>
RangeIndex: 623 entries, 0 to 622
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   loan_id      623 non-null    str    
 1   member_id    623 non-null    str    
 2   title        623 non-null    str    
 3   genre        623 non-null    str    
 4   loan_date    623 non-null    str    
 5   return_date  593 non-null    str    
 6   late_fee     623 non-null    str    
 7   returned     623 non-null    str    
 8   notes        0 non-null      float64
dtypes: float64(1), str(8)
memory usage: 43.9 KB


The raw table has 623 rows and 9 columns. notes is completely empty. loan_date and return_date are text rather than dates; late_fee is text rather than a number; returned is text rather than a Boolean. return_date has 593 non-missing values and 30 missing values.

## 2. Clean the genre

In [5]:
loans['genre'].nunique()

28

In [6]:
loans['genre'] = loans['genre'].str.strip().str.lower()

loans['genre'].nunique()

9

In [7]:
loans['genre'].unique()

<StringArray>
[        'romance',           'crime',         'fantasy',        'children',
      'children's',         'history',          'sci-fi', 'science fiction',
          'sci fi']
Length: 9, dtype: str

In [8]:
loans['genre'] = loans['genre'].replace({
    "children's": 'children',
    'sci fi': 'sci-fi',
    'science fiction': 'sci-fi'
})
loans['genre'] = loans['genre'].str.title()

In [9]:
loans['genre'].nunique()

6

In [10]:
loans['genre'].value_counts()

genre
Crime       163
Children    135
Fantasy      90
Sci-Fi       90
Romance      86
History      59
Name: count, dtype: int64

There are 28 original values, 9 after stripping spaces and lowercasing, and 6 after fixing the remaining variants. Counts: Crime 163, Children 135, Fantasy 90, Sci-Fi 90, Romance 86, History 59.

## 3. Yes and no

In [11]:
loans['returned'].nunique()

10

In [12]:
loans['returned'] = loans['returned'].str.strip().str.lower()
loans['returned'] = loans['returned'].map({
    'yes': True, 'y': True, 'no': False, 'n': False
})

In [13]:
loans['returned'].isna().sum()

np.int64(0)

In [14]:
loans['returned'].value_counts()

returned
True     593
False     30
Name: count, dtype: int64

The 10 original values become True (593) and False (30), with no missing values after mapping.

## 4. The late fee is text

Intentional error: try converting the original text directly.

In [15]:
loans['late_fee'].astype(float)

ValueError: could not convert string to float: '7,50 kr'

Remove the currency text and spaces, then change the decimal comma to a dot.

In [16]:
loans['late_fee'] = loans['late_fee'].str.replace('kr', '', regex=False)
loans['late_fee'] = loans['late_fee'].str.replace(' ', '', regex=False)
loans['late_fee'] = loans['late_fee'].str.replace(',', '.', regex=False)
loans['late_fee'] = loans['late_fee'].astype(float)

In [17]:
loans['late_fee'].describe().round(2)

count    623.00
mean      12.32
std       16.11
min        0.00
25%        0.00
50%        0.00
75%       25.00
max       47.50
Name: late_fee, dtype: float64

In [18]:
loans['late_fee'].sum()

np.float64(7677.5)

The direct conversion fails because values such as 7,50 kr are not valid float strings. After cleaning: minimum 0.0, maximum 47.5, mean 12.32, and total 7,677.5 kr.

## 5. Dates

In [19]:
loans['loan_date'] = pd.to_datetime(loans['loan_date'], errors='coerce')

In [20]:
loans['loan_date'].isna().sum()

np.int64(3)

In [21]:
print(loans['loan_date'].min())
print(loans['loan_date'].max())

2026-01-05 00:00:00
2026-06-27 00:00:00


In [22]:
monthly_loans = loans['loan_date'].dt.to_period('M').value_counts().sort_index()

monthly_loans

loan_date
2026-01     96
2026-02    100
2026-03    111
2026-04    106
2026-05    108
2026-06     99
Freq: M, Name: count, dtype: int64

In [ ]:
print(monthly_loans.sum())
print(len(loans))
print(loans['loan_date'].isna().sum())

Three invalid dates become NaT. The valid dates range from 2026-01-05 to 2026-06-27. Monthly counts are 96, 100, 111, 106, 108, and 99. Their total is 620, so the 3 missing dates account for the difference from 623 rows.

## 6. Duplicates

In [23]:
loans_raw.duplicated().sum()

np.int64(12)

In [24]:
loans.duplicated().sum()

np.int64(18)

In [25]:
loans.duplicated(subset='loan_id').sum()

np.int64(18)

There are 12 duplicate rows before cleaning and 18 after cleaning. Cleaning makes 6 more rows match because their spelling or formatting is now the same. Checking loan_id also finds 18 duplicates.

In [26]:
fees_before = loans['late_fee'].sum()

loans = loans.drop_duplicates().reset_index(drop=True)

print(loans.shape)

(605, 9)


In [27]:
fees_after = loans['late_fee'].sum()
extra_fees = fees_before - fees_after

print(fees_before)
print(fees_after)
print(extra_fees)
print(round(extra_fees / fees_before * 100, 2))

7677.5
7447.5
230.0
3.0


After removing duplicates there are 605 rows and 9 columns. Fees fall from 7,677.5 to 7,447.5 kr. The duplicates added 230 kr, about 3.0% of the total before removing them.

## 7. Put it together

In [28]:
loans = loans.drop(columns='notes')

loans.shape

(605, 8)

All cleaning steps in one cell. Run this cell twice to check that it gives the same result.

In [29]:
loans = loans_raw.copy()

loans['genre'] = loans['genre'].str.strip().str.lower()
loans['genre'] = loans['genre'].replace({
    "children's": 'children',
    'sci fi': 'sci-fi',
    'science fiction': 'sci-fi'
})
loans['genre'] = loans['genre'].str.title()

loans['returned'] = loans['returned'].str.strip().str.lower()
loans['returned'] = loans['returned'].map({
    'yes': True, 'y': True, 'no': False, 'n': False
})

loans['late_fee'] = loans['late_fee'].str.replace('kr', '', regex=False)
loans['late_fee'] = loans['late_fee'].str.replace(' ', '', regex=False)
loans['late_fee'] = loans['late_fee'].str.replace(',', '.', regex=False)
loans['late_fee'] = loans['late_fee'].astype(float)

loans['loan_date'] = pd.to_datetime(loans['loan_date'], errors='coerce')

loans = loans.drop_duplicates().reset_index(drop=True)
loans = loans.drop(columns='notes')

print(loans.shape)
loans.to_csv('data/loans_clean.csv', index=False)

(605, 8)


The final shape should be (605, 8). The combined cell works on repeated runs because it starts from a fresh copy of loans_raw every time. return_date conversion is introduced in Part 2.